# 04. Perfilado del Dataset Unificado

Este notebook analiza el estado del dataset tras la integración vertical de las fuentes interna y externa.

**Objetivo:**
1. Generar un reporte de calidad automático para el dataset unificado.
2. Identificar el grado de duplicidad nominal (mismo nombre/correo) antes de aplicar Record Linkage probabilístico.

**Nota Importante:** En este notebook NO eliminamos duplicados. Solo diagnosticamos. La consolidación real ocurre en el Notebook 05 para permitir una auditoría completa de registros fusionados y huérfanos.

In [1]:
import pandas as pd
import os
import json
import sweetviz as sv
import warnings
warnings.filterwarnings('ignore')

# Configuración de rutas
PROCESSED_PATH = '../data/processed/'
RESULTS_PATH = '../data/results/'

# Carga del dataset unificado
df_unified = pd.read_csv(os.path.join(PROCESSED_PATH, 'unified_dataset.csv'))

print(f"Dataset unificado cargado: {df_unified.shape[0]} registros, {df_unified.shape[1]} columnas")

C:\Users\erick\Desktop\hotel-booking-ops\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset unificado cargado: 238780 registros, 34 columnas


## 1. Reporte de Perfilado Automático
Generamos el reporte integral para detectar inconsistencias remanentes tras la limpieza.

In [2]:
report = sv.analyze(df_unified)
report.show_html(os.path.join(RESULTS_PATH, 'perfilado_integrado.html'), open_browser=False)

print("[OK] Reporte 'perfilado_integrado.html' exportado.")

Done! Use 'show' commands to display/save.   |██████████| [100%]   00:00 -> (00:00 left)

Report ../data/results/perfilado_integrado.html was generated.
[OK] Reporte 'perfilado_integrado.html' exportado.


## 2. Diagnóstico de Duplicados Exactos (Ignorando 'source')
Identificamos registros que son idénticos en todas sus dimensiones de negocio.

In [3]:
# Definimos las columnas de negocio (todas menos 'source')
cols_negocio = [c for c in df_unified.columns if c != 'source']

num_duplicados_exactos = df_unified.duplicated(subset=cols_negocio).sum()
print(f"Registros duplicados exactos (en dimensiones de negocio) detectados: {num_duplicados_exactos}")
print("[INFO] No se eliminan registros en esta fase para permitir auditoría en el Notebook 05.")

Registros duplicados exactos (en dimensiones de negocio) detectados: 10243
[INFO] No se eliminan registros en esta fase para permitir auditoría en el Notebook 05.


## 3. Análisis de Colisiones de Identidad
Buscamos colisiones en campos clave (Nombre, Email, Teléfono).

In [4]:
# Colisiones por Email (excluyendo nulos)
email_counts = df_unified.dropna(subset=['email'])['email'].value_counts()
candidatos_email = email_counts[email_counts > 1].sum()

# Colisiones por Teléfono
phone_counts = df_unified.dropna(subset=['phone_number'])['phone_number'].value_counts()
candidatos_phone = phone_counts[phone_counts > 1].sum()

print(f"Registros con email compartido: {candidatos_email}")
print(f"Registros con teléfono compartido: {candidatos_phone}")

Registros con email compartido: 227380
Registros con teléfono compartido: 227018


## 4. Métricas de Unicidad Nominal
Calculamos la unicidad inicial basada en identificadores.

In [5]:
unicidad_pre = (1 - (df_unified.duplicated(subset=['name', 'email', 'phone_number']).sum() / len(df_unified))) * 100
print(f"Unicidad nominal estimada: {unicidad_pre:.2f}%")
print("\n[FIN] El dataset permanece intacto para la fase de consolidación probabilística.")

Unicidad nominal estimada: 86.47%

[FIN] El dataset permanece intacto para la fase de consolidación probabilística.
